Once raw data fills your Bronze layer, run your cleaning notebook. It reads the raw files from your Bronze volume, bursts open the arrays, runs your recursive flattener, and writes a clean delta table straight into your Silver layer container.

In [0]:
# Databricks notebook source
from pyspark.sql import functions as F
from pyspark.sql.types import StructType

# Setup storage paths
bronze_volume_path = "/Volumes/weather_catalog/bronze/raw_forecasts/*/*.json"
silver_table_path  = "abfss://silver@stweatherprojectnew.dfs.core.windows.net/forecasts"

# 1. Recursive Flattener Function
def flatten_dataframe(df):
    complex_fields = dict([(field.name, field.dataType) for field in df.schema if isinstance(field.dataType, StructType)])
    while len(complex_fields) > 0:
        col_name = list(complex_fields.keys())[0]
        sub_fields = [field.name for field in complex_fields[col_name].fields]
        select_expr = [F.col(f"`{col_name}`.`{field}`").alias(f"{col_name}_{field}") for field in sub_fields]
        other_cols = [F.col(f"`{c}`") for c in df.columns if c != col_name]
        df = df.select(*other_cols, *select_expr)
        complex_fields = dict([(field.name, field.dataType) for field in df.schema if isinstance(field.dataType, StructType)])
    return df

# 2. Read raw multi-line JSON files out of Bronze volume
print("Reading raw files from Bronze...")
df_raw = spark.read.option("multiline", "true").json(bronze_volume_path)

# 3. Handle AccuWeather array nesting by exploding the DailyForecasts list
if "DailyForecasts" in df_raw.columns:
    df_exploded = df_raw.withColumn("DailyForecasts", F.explode("DailyForecasts"))
else:
    df_exploded = df_raw

# 4. Clean data by running recursive flattening
cleaned_df = flatten_dataframe(df_exploded)
#display(cleaned_df)
# # 5. Create Silver Metadata schema and save the table
# spark.sql("CREATE SCHEMA IF NOT EXISTS weather_catalog.silver")

# cleaned_df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .option("path", silver_table_path) \
#     .option("overwriteSchema", "true") \
#     .saveAsTable("weather_catalog.silver.forecasts")

# print("Data successfully cleaned and stored as a Delta table in Silver.")

Reading raw files from Bronze...


LocationKey,DailyForecasts_AirAndPollen,DailyForecasts_Date,DailyForecasts_EpochDate,DailyForecasts_HoursOfSun,DailyForecasts_Link,DailyForecasts_MobileLink,DailyForecasts_Sources,Headline_Category,Headline_EffectiveDate,Headline_EffectiveEpochDate,Headline_EndDate,Headline_EndEpochDate,Headline_Link,Headline_MobileLink,Headline_Severity,Headline_Text,DailyForecasts_Day_CloudCover,DailyForecasts_Day_HasPrecipitation,DailyForecasts_Day_HoursOfIce,DailyForecasts_Day_HoursOfPrecipitation,DailyForecasts_Day_HoursOfRain,DailyForecasts_Day_HoursOfSnow,DailyForecasts_Day_IceProbability,DailyForecasts_Day_Icon,DailyForecasts_Day_IconPhrase,DailyForecasts_Day_LongPhrase,DailyForecasts_Day_PrecipitationIntensity,DailyForecasts_Day_PrecipitationProbability,DailyForecasts_Day_PrecipitationType,DailyForecasts_Day_RainProbability,DailyForecasts_Day_ShortPhrase,DailyForecasts_Day_SnowProbability,DailyForecasts_Day_ThunderstormProbability,DailyForecasts_Moon_Age,DailyForecasts_Moon_EpochRise,DailyForecasts_Moon_EpochSet,DailyForecasts_Moon_Phase,DailyForecasts_Moon_Rise,DailyForecasts_Moon_Set,DailyForecasts_Night_CloudCover,DailyForecasts_Night_HasPrecipitation,DailyForecasts_Night_HoursOfIce,DailyForecasts_Night_HoursOfPrecipitation,DailyForecasts_Night_HoursOfRain,DailyForecasts_Night_HoursOfSnow,DailyForecasts_Night_IceProbability,DailyForecasts_Night_Icon,DailyForecasts_Night_IconPhrase,DailyForecasts_Night_LongPhrase,DailyForecasts_Night_PrecipitationIntensity,DailyForecasts_Night_PrecipitationProbability,DailyForecasts_Night_PrecipitationType,DailyForecasts_Night_RainProbability,DailyForecasts_Night_ShortPhrase,DailyForecasts_Night_SnowProbability,DailyForecasts_Night_ThunderstormProbability,DailyForecasts_Sun_EpochRise,DailyForecasts_Sun_EpochSet,DailyForecasts_Sun_Rise,DailyForecasts_Sun_Set,DailyForecasts_Day_Evapotranspiration_Unit,DailyForecasts_Day_Evapotranspiration_UnitType,DailyForecasts_Day_Evapotranspiration_Value,DailyForecasts_Day_Ice_Unit,DailyForecasts_Day_Ice_UnitType,DailyForecasts_Day_Ice_Value,DailyForecasts_Day_Rain_Unit,DailyForecasts_Day_Rain_UnitType,DailyForecasts_Day_Rain_Value,DailyForecasts_Day_RelativeHumidity_Average,DailyForecasts_Day_RelativeHumidity_Maximum,DailyForecasts_Day_RelativeHumidity_Minimum,DailyForecasts_Day_Snow_Unit,DailyForecasts_Day_Snow_UnitType,DailyForecasts_Day_Snow_Value,DailyForecasts_Day_SolarIrradiance_Unit,DailyForecasts_Day_SolarIrradiance_UnitType,DailyForecasts_Day_SolarIrradiance_Value,DailyForecasts_Day_TotalLiquid_Unit,DailyForecasts_Day_TotalLiquid_UnitType,DailyForecasts_Day_TotalLiquid_Value,DailyForecasts_Day_UVIndexFloat_Maximum,DailyForecasts_Day_UVIndexFloat_Minimum,DailyForecasts_DegreeDaySummary_Cooling_Unit,DailyForecasts_DegreeDaySummary_Cooling_UnitType,DailyForecasts_DegreeDaySummary_Cooling_Value,DailyForecasts_DegreeDaySummary_Heating_Unit,DailyForecasts_DegreeDaySummary_Heating_UnitType,DailyForecasts_DegreeDaySummary_Heating_Value,DailyForecasts_Night_Evapotranspiration_Unit,DailyForecasts_Night_Evapotranspiration_UnitType,DailyForecasts_Night_Evapotranspiration_Value,DailyForecasts_Night_Ice_Unit,DailyForecasts_Night_Ice_UnitType,DailyForecasts_Night_Ice_Value,DailyForecasts_Night_Rain_Unit,DailyForecasts_Night_Rain_UnitType,DailyForecasts_Night_Rain_Value,DailyForecasts_Night_RelativeHumidity_Average,DailyForecasts_Night_RelativeHumidity_Maximum,DailyForecasts_Night_RelativeHumidity_Minimum,DailyForecasts_Night_Snow_Unit,DailyForecasts_Night_Snow_UnitType,DailyForecasts_Night_Snow_Value,DailyForecasts_Night_SolarIrradiance_Unit,DailyForecasts_Night_SolarIrradiance_UnitType,DailyForecasts_Night_SolarIrradiance_Value,DailyForecasts_Night_TotalLiquid_Unit,DailyForecasts_Night_TotalLiquid_UnitType,DailyForecasts_Night_TotalLiquid_Value,DailyForecasts_Night_UVIndexFloat_Maximum,DailyForecasts_Night_UVIndexFloat_Minimum,DailyForecasts_RealFeelTemperature_Maximum_Phrase,DailyForecasts_RealFeelTemperature_Maximum_Unit,DailyForecasts_RealFeelTemperature_Maximum_Uni

In [0]:
final_forecast_data = cleaned_df.select("LocationKey","DailyForecasts_Date","Headline_EffectiveDate","Headline_EndDate","Headline_Text","DailyForecasts_Day_HoursOfRain","DailyForecasts_Night_HoursOfRain","DailyForecasts_Day_ShortPhrase","DailyForecasts_Day_LongPhrase","DailyForecasts_Day_ThunderstormProbability","DailyForecasts_Night_LongPhrase","DailyForecasts_Night_ShortPhrase","DailyForecasts_Night_ThunderstormProbability","DailyForecasts_Day_Rain_Unit","DailyForecasts_Night_Rain_Unit","DailyForecasts_Day_Rain_Value","DailyForecasts_Night_Rain_Value")


LocationKey,DailyForecasts_Date,Headline_EffectiveDate,Headline_EndDate,Headline_Text,DailyForecasts_Day_HoursOfRain,DailyForecasts_Night_HoursOfRain,DailyForecasts_Day_ShortPhrase,DailyForecasts_Day_LongPhrase,DailyForecasts_Day_ThunderstormProbability,DailyForecasts_Night_HoursOfRain,DailyForecasts_Night_LongPhrase,DailyForecasts_Night_ShortPhrase,DailyForecasts_Night_ThunderstormProbability,DailyForecasts_Day_Rain_Unit,DailyForecasts_Night_Rain_Unit,DailyForecasts_Day_Rain_Value,DailyForecasts_Night_Rain_Value
202396,2026-06-26T07:00:00+05:30,2026-06-26T19:00:00+05:30,2026-06-28T07:00:00+05:30,Air quality will be very unhealthy Friday evening through late Saturday night,0.0,0.0,Hazy sunshine and very hot,Hazy sunshine and very hot; danger of dehydration and heatstroke if outside for extended periods of time,6,0.0,Very warm with a crystal-clear sky; air quality will be very unhealthy,Clear and very warm,1,mm,mm,0.0,0.0
202396,2026-06-27T07:00:00+05:30,2026-06-26T19:00:00+05:30,2026-06-28T07:00:00+05:30,Air quality will be very unhealthy Friday evening through late Saturday night,0.0,0.0,Very warm with hazy sunshine,Very warm with hazy sunshine; air quality will be very unhealthy,1,0.0,Very warm with a crystal-clear sky; air quality will be very unhealthy,Clear and very warm,0,mm,mm,0.0,0.0
202396,2026-06-28T07:00:00+05:30,2026-06-26T19:00:00+05:30,2026-06-28T07:00:00+05:30,Air quality will be very unhealthy Friday evening through late Saturday night,1.0,1.0,Sweltering heat,Hazy sunshine and sweltering heat; a thunderstorm around in the afternoon; dangerous heat,33,1.0,Mainly clear and very warm; a thunderstorm late; air quality will be very unhealthy,Warm; a thunderstorm late,33,mm,mm,1.0,1.3
202396,2026-06-29T07:00:00+05:30,2026-06-26T19:00:00+05:30,2026-06-28T07:00:00+05:30,Air quality will be very unhealthy Friday evening through late Saturday night,0.5,1.0,A strong afternoon t-storm,Mostly sunny and sweltering heat; watch for a strong thunderstorm in the afternoon; dangerous heat,52,1.0,Mainly clear and very warm; a thunderstorm in spots late; air quality will be very unhealthy,Warm; a t-storm around late,33,mm,mm,0.5,1.0
202396,2026-06-30T07:00:00+05:30,2026-06-26T19:00:00+05:30,2026-06-28T07:00:00+05:30,Air quality will be very unhealthy Friday evening through late Saturday night,0.0,1.5,Hazy sun and sweltering heat,Hazy sun and sweltering heat; air quality will be very unhealthy,1,1.5,Mainly clear and very warm; a thunderstorm in spots in the evening followed by a shower in spots late; air quality will be very unhealthy,A thunderstorm around early,33,mm,mm,0.0,1.0
204842,2026-06-26T07:00:00+05:30,2026-06-26T13:00:00+05:30,2026-07-01T07:00:00+05:30,Expect rainy weather Friday afternoon through late Tuesday night,2.5,2.5,"Mostly cloudy, a little rain",Mainly cloudy with a bit of rain,18,2.5,Overcast with a touch of rain,Overcast with a touch of rain,18,mm,mm,2.2,4.7
204842,2026-06-27T07:00:00+05:30,2026-06-26T13:00:00+05:30,2026-07-01T07:00:00+05:30,Expect rainy weather Friday afternoon through late Tuesday night,2.5,3.5,"Mostly cloudy, a little rain",Rather cloudy with a bit of rain,14,3.5,Overcast with a touch of rain,Overcast with a touch of rain,15,mm,mm,3.5,8.2
204842,2026-06-28T07:00:00+05:30,2026-06-26T13:00:00+05:30,2026-07-01T07:00:00+05:30,Expect rainy weather Friday afternoon through late Tuesday night,3.5,6.5,Periods of rain,Periods of rain,22,6.5,Rain,Rain,23,mm,mm,5.9,15.0
204842,2026-06-29T07:00:00+05:30,2026-06-26T13:00:00+05:30,2026-07-01T07:00:00+05:30,Expect rainy weather Friday afternoon through late Tuesday night,6.0,7.5,Rain; breezy in the afternoon,Rain; breezy in the afternoon,23,7.5,Rain; breezy in the evening,Rain; breezy in the evening,23,mm,mm,14.4,15.1
204842,2026-06-30T07:00:00+05:30,2026-06-26T13:00:00+05:30,2026-07-01T07:00:00+05:30,Expect rainy weather Friday afternoon through late Tuesday night,7.0,12.0,Breezy with rain,Breezy with rain,23,12.0,Rain; breezy in the evening,Rain; breezy i

In [0]:
# 5. Create Silver Metadata schema and save the table
spark.sql("CREATE SCHEMA IF NOT EXISTS weather_catalog.silver")

final_forecast_data.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", silver_table_path) \
    .option("overwriteSchema", "true") \
    .saveAsTable("weather_catalog.silver.forecasts")

print("Data successfully cleaned and stored as a Delta table in Silver.")

Data successfully cleaned and stored as a Delta table in Silver.
